### Find obsidian non-note files not linked to by an obsidian note.
Gets all links in Obsidian notes, and looks for non-note files which aren't linked to
Uses file regexps, instead of using the prohibitively slow obsidiantools or py-obsidianmd libs

##### TODO
- [ ] only go through lit_sources and lit_notes

In [ ]:
import os
import re
import pathlib as pl
import pandas as pd
from icecream import ic
from collections import defaultdict
import matplotlib.pyplot as plt
import refwrangle as rfw

In [1]:

def find_all_links(vault_path):
    """Find all links inside of all obsidian .md note files"""
    link_pattern = re.compile(r'\[([^\]]+)\]\(([^\)]+)\)|(!?)\[\[([^\]|#]+)(?:#([^\]|]+))?(?:\|([^\]]+))?\]\]')
    exclude_dirs = ['.makemd','.md','.obsidian','.smart-env','trash','templates','Scratch Space',
                    'HTML','HTML import','Evernote']
    
    links = []
    non_note_files = defaultdict(list)
    for root, dirs, files in os.walk(vault_path, topdown=True): # topdown so dirs mode below works
        dirs[:] = [d for d in dirs if d not in exclude_dirs]  # filtering dirs in-place limits files looped below

        for file in files:
            if file.endswith('.md'):
                # get the links
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    file_links = link_pattern.findall(content)
                    for link in file_links:
                        if link[0] and link[1]:  # Markdown link
                            ltype, srcpath, ltext, ltarg = 'markdown', file_path, link[0], link[1]  
                            is_image, section = False,''  # is_image is BOGUS here
                        elif link[3]:  # Wiki link or image
                            is_image = bool(link[2])  # True if prefixed with '!'
                            link_target = link[3]
                            section = link[4] if link[4] else ""
                            link_text = link[5] if link[5] else (link[3] + ("#" + section if section else ""))    
                            ltype, srcpath, ltext, ltarg = 'wiki', file_path, link_text, link_target

                        links.append(dict(ltype=ltype, lSrcPath=srcpath, ltext=ltext, ltargFNm=ltarg, lparts=link, isTargImage=is_image, targSection=section))
            else:                        
                # store the filename
                baseFNm = os.path.basename(file)
                file_fullpath = os.path.join(root, file)
                non_note_files[baseFNm].append(file_fullpath)

    return pd.DataFrame(links), non_note_files


In [ ]:
all_links, non_note_files = find_all_links(rfw.obsidian_vault_dir)
print(f'Found {len(all_links)} links and {len(non_note_files)} non_note_file base file names')

In [ ]:
repeated_non_notes = [key for key, value in non_note_files.items() if len(value) > 1]
if (nRepeats := len(repeated_non_notes)) > 0:
    print(f"{nRepeats} repeated non-note base file names:")

    for baseFNm in repeated_non_notes:
        fullFNms = non_note_files[baseFNm]
        print(f"{len(fullFNms)}: {baseFNm}")
        for fullFNm in fullFNms:
            print(f"\t{fullFNm}")

In [ ]:
# find unlinked files
non_note_baseFNms_no_link = list(set(non_note_files.keys()) - set(all_links.ltargFNm))

print(f'{len(non_note_baseFNms_no_link)} of {len(non_note_files.keys())} non-note files are unlinked')

non_note_no_link_info = pd.DataFrame([(baseFNm, pl.Path(baseFNm).stem, pl.Path(baseFNm).suffix) 
                              for baseFNm in non_note_baseFNms_no_link], columns=['fNm', 'basename', 'extension'])
(non_note_no_link_info.extension.value_counts().sort_values(ascending=True)
 .plot(kind='barh', color='skyblue', edgecolor='black', xlabel='Count', title='Unlinked File Types'));

In [ ]:
for k, v in all_links.query('ltype=="markdown"').iloc[200].items():
    print(f'{k}: {v}')

In [ ]:
all_links.query('ltype=="wiki"')